# Pipeline de avaliação — multi-modelo × multi-edital

Roda o agente em todas as combinações `(modelo, edital)` definidas, salva um xlsx
por combinação em `resultados/` e exibe sumário consolidado.

**Proteções contra desastre:**
1. Validação de preços antes de rodar (pega `PRECOS` faltando).
2. Smoke test: 1 pergunta por combinação antes de soltar as 50.
3. Limite de gasto por combinação — corta no meio se estourar.
4. Salvamento garantido em qualquer erro: o que foi medido até a falha vai pro xlsx.
5. Sumário final lista sucessos, parciais e falhas separadamente.

In [11]:
import os, sys, time, glob
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from agent.agent import build_agent, ask
from langchain_core.callbacks import BaseCallbackHandler

print(f"cwd:  {Path.cwd()}")
print(f"root: {ROOT}")

cwd:  /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/evals
root: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant


## TokenTracker — captura tokens + cache

In [12]:
class TokenTracker(BaseCallbackHandler):
    def __init__(self):
        self.reset()

    def reset(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.cache_read_tokens = 0
        self.cache_write_tokens = 0
        self.n_calls = 0

    def on_llm_end(self, response, **kwargs):
        self.n_calls += 1

        usage = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
        if usage:
            self.input_tokens  += usage.get("prompt_tokens", 0) or 0
            self.output_tokens += usage.get("completion_tokens", 0) or 0
            ds_hit = usage.get("prompt_cache_hit_tokens") or 0
            details = usage.get("prompt_tokens_details") or {}
            oai_hit = details.get("cached_tokens") or 0
            self.cache_read_tokens += (ds_hit + oai_hit)
            return

        for gen_list in response.generations:
            for gen in gen_list:
                msg = getattr(gen, "message", None)
                um = getattr(msg, "usage_metadata", None) if msg else None
                if um:
                    self.input_tokens  += um.get("input_tokens", 0) or 0
                    self.output_tokens += um.get("output_tokens", 0) or 0
                    details = um.get("input_token_details") or {}
                    self.cache_read_tokens  += details.get("cache_read", 0) or 0
                    self.cache_write_tokens += details.get("cache_creation", 0) or 0
                    break

## Configuração

Comente os índices que não quer rodar. `LIMITE_USD_POR_COMBO` é teto sobre estimativa
pessimista (sem cache) — real será sempre menor.

In [13]:
MODELOS = [
    # (provider, modelo)
    ("openai",    "gpt-4o-mini"),
    ("deepseek",  "deepseek-v4-flash"),
    ("openai",    "gpt-5.4-mini"),
    ("deepseek",  "deepseek-v4-pro"),
    ("anthropic", "claude-haiku-4-5"),
    # --- modelos caros — descomente quando for rodar ---
    # ("openai",    "gpt-5.4"),
    # ("anthropic", "claude-sonnet-4-6"),
    # ("anthropic", "claude-opus-4-7"),
    # ("openai",    "gpt-5.5"),
]

EDITAIS = [
    ("bndes",     1),
    ("cvm",       2),
    ("petrobras", 3),
]

# USD por 1M tokens. Ordem: (input_miss, input_cached, output).
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.075,  0.60),
    "openai/gpt-5.4-mini":            (0.25,  0.125,  2.00),
    "openai/gpt-5.4":                 (1.25,  0.625, 10.00),
    "openai/gpt-5.5":                 (5.00,  2.500, 30.00),
    "deepseek/deepseek-v4-flash":     (0.14,  0.028,  0.28),
    "deepseek/deepseek-v4-pro":       (1.74,  0.145,  3.48),
    "anthropic/claude-haiku-4-5":     (1.00,  1.000,  5.00),
    "anthropic/claude-sonnet-4-6":    (3.00,  3.000, 15.00),
    "anthropic/claude-opus-4-7":      (5.00,  5.000, 25.00),
}

LIMITE_USD_POR_COMBO = 20.00

PERGUNTAS_DIR  = Path("perguntas")
RESULTADOS_DIR = Path("resultados")
RESULTADOS_DIR.mkdir(exist_ok=True)

print(f"Modelos ativos:        {len(MODELOS)}")
print(f"Editais:               {len(EDITAIS)}")
print(f"Total de combinações:  {len(MODELOS) * len(EDITAIS)}")
print(f"Teto por combinação:   ${LIMITE_USD_POR_COMBO}")

Modelos ativos:        5
Editais:               3
Total de combinações:  15
Teto por combinação:   $20.0


## Validação 1 — preços (gratuita, não chama API)

In [14]:
faltando = [f"{p}/{m}" for p, m in MODELOS if f"{p}/{m}" not in PRECOS]
if faltando:
    raise RuntimeError(f"Modelos sem preço em PRECOS: {faltando}")
print(f"OK — todos os {len(MODELOS)} modelos têm preço configurado.")

OK — todos os 5 modelos têm preço configurado.


## Helpers

In [15]:
def slug(modelo: str) -> str:
    return modelo.replace("/", "_").replace(":", "_")


def custo_estimado(in_tok: int, out_tok: int, p_in: float, p_out: float) -> float:
    """Teto: trata todo input como cache miss."""
    return in_tok / 1_000_000 * p_in + out_tok / 1_000_000 * p_out


def custo_real(in_tok: int, out_tok: int, cache_read: int,
               p_in: float, p_in_cached: float, p_out: float) -> float:
    miss = max(in_tok - cache_read, 0)
    return (
        miss / 1_000_000 * p_in
        + cache_read / 1_000_000 * p_in_cached
        + out_tok / 1_000_000 * p_out
    )


def salvar_xlsx(provider, modelo, edital_nome, resultados, sufixo=""):
    df_r = pd.DataFrame(resultados)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    nome = f"{edital_nome}__{provider}__{slug(modelo)}__{ts}{sufixo}.xlsx"
    arq = RESULTADOS_DIR / nome
    df_r.to_excel(arq, index=False)
    return arq


def apagar_antigos(provider, modelo, edital_nome):
    padrao = str(RESULTADOS_DIR / f"{edital_nome}__{provider}__{slug(modelo)}__*.xlsx")
    n = 0
    for antigo in glob.glob(padrao):
        Path(antigo).unlink()
        n += 1
    return n

In [16]:
def rodar_combo(provider, modelo, edital_nome, edital_id, limite_usd=LIMITE_USD_POR_COMBO):
    """Roda N perguntas. Retorna (xlsx_path, erro_fatal_ou_None).
    
    Erros em perguntas individuais vão pra coluna `erro` e a combinação continua.
    Estouro de orçamento ou erro fora do loop interrompem e salvam parcial."""
    chave = f"{provider}/{modelo}"
    p_in, p_in_cached, p_out = PRECOS[chave]

    arq_perguntas = PERGUNTAS_DIR / f"{edital_nome}.xlsx"
    df_q = pd.read_excel(arq_perguntas)
    df_q = df_q.dropna(subset=["pergunta"])
    df_q = df_q[df_q["pergunta"].str.strip() != ""].reset_index(drop=True)

    agente = build_agent(provider=provider, model=modelo)
    tracker = TokenTracker()
    agente.llm       = agente.llm.with_config(callbacks=[tracker])
    agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

    resultados = []
    custo_estim_acum = 0.0
    erro_fatal = None

    for i, row in df_q.iterrows():
        pergunta = row["pergunta"]
        t0 = time.time()
        erro, resposta = None, None

        tracker.reset()
        try:
            resposta = ask(agent=agente, question=pergunta, chat_history=[], edital_id_ativo=edital_id)
        except Exception as e:
            erro = str(e)

        in_tok     = tracker.input_tokens
        out_tok    = tracker.output_tokens
        cache_read = tracker.cache_read_tokens
        n_calls    = tracker.n_calls
        latencia   = round(time.time() - t0, 2)

        c_estim = custo_estimado(in_tok, out_tok, p_in, p_out)
        c_real  = custo_real(in_tok, out_tok, cache_read, p_in, p_in_cached, p_out)
        custo_estim_acum += c_estim

        resultados.append({
            "id":                row["id"],
            "categoria":         row.get("categoria", ""),
            "pergunta":          pergunta,
            "resposta":          resposta,
            "input_tokens":      in_tok,
            "output_tokens":     out_tok,
            "total_tokens":      in_tok + out_tok,
            "cache_read_tokens": cache_read,
            "cache_miss_tokens": max(in_tok - cache_read, 0),
            "n_invocacoes":      n_calls,
            "custo_usd":         round(c_estim, 6),
            "custo_real_usd":    round(c_real, 6),
            "latencia_s":        latencia,
            "erro":              erro,
        })

        flag = "ERRO" if erro else "ok"
        cache_pct = (cache_read / in_tok * 100) if in_tok else 0
        print(f"      [{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
              f"calls={n_calls}  in={in_tok:>5} (cache {cache_pct:>4.0f}%)  "
              f"out={out_tok:>4}  acum=${custo_estim_acum:.4f}")

        if custo_estim_acum > limite_usd:
            erro_fatal = f"Estouro na pergunta {i+1}: ${custo_estim_acum:.4f} > ${limite_usd}"
            break

    sufixo = "__PARCIAL" if erro_fatal else ""
    arq = salvar_xlsx(provider, modelo, edital_nome, resultados, sufixo) if resultados else None
    return arq, erro_fatal

## Validação 2 — smoke test

Uma pergunta por combinação. Se algum modelo tiver armadilha (thinking mode default,
credencial faltando, formato de tool diferente), explode aqui em vez de em 50
perguntas. Se qualquer combo falhar, **a célula levanta erro** e o loop principal
não roda.

In [17]:
print("=== SMOKE TEST ===\n")

smoke_falhas = []

for provider, modelo in MODELOS:
    for edital_nome, edital_id in EDITAIS:
        print(f"  [test] {provider}/{modelo} × {edital_nome} ... ", end="", flush=True)
        try:
            agente = build_agent(provider=provider, model=modelo)
            tracker = TokenTracker()
            agente.llm       = agente.llm.with_config(callbacks=[tracker])
            agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

            resp = ask(agent=agente, question="Qual o salário inicial?",
                       chat_history=[], edital_id_ativo=edital_id)

            if not resp or len(resp.strip()) < 20:
                raise ValueError(f"resposta vazia/curta: {resp!r}")
            print(f"ok ({tracker.input_tokens} in / {tracker.output_tokens} out / {tracker.n_calls} calls)")
        except Exception as e:
            smoke_falhas.append(f"{provider}/{modelo} × {edital_nome}: {e}")
            print(f"FALHOU\n           {e}")

if smoke_falhas:
    raise RuntimeError(
        f"\n{len(smoke_falhas)} combinação(ões) falhou(ram). Corrija ou comente:\n  - " +
        "\n  - ".join(smoke_falhas)
    )

print(f"\nTodas as {len(MODELOS) * len(EDITAIS)} combinações passaram.")

=== SMOKE TEST ===

  [test] openai/gpt-4o-mini × bndes ... ok (7590 in / 105 out / 2 calls)
  [test] openai/gpt-4o-mini × cvm ... ok (7146 in / 88 out / 2 calls)
  [test] openai/gpt-4o-mini × petrobras ... ok (7251 in / 78 out / 2 calls)
  [test] deepseek/deepseek-v4-flash × bndes ... ok (9569 in / 205 out / 2 calls)
  [test] deepseek/deepseek-v4-flash × cvm ... ok (8547 in / 160 out / 2 calls)
  [test] deepseek/deepseek-v4-flash × petrobras ... ok (9177 in / 204 out / 2 calls)
  [test] openai/gpt-5.4-mini × bndes ... ok (5150 in / 90 out / 2 calls)
  [test] openai/gpt-5.4-mini × cvm ... ok (4523 in / 104 out / 2 calls)
  [test] openai/gpt-5.4-mini × petrobras ... ok (4564 in / 89 out / 2 calls)
  [test] deepseek/deepseek-v4-pro × bndes ... ok (9128 in / 200 out / 2 calls)
  [test] deepseek/deepseek-v4-pro × cvm ... ok (8539 in / 162 out / 2 calls)
  [test] deepseek/deepseek-v4-pro × petrobras ... ok (9237 in / 184 out / 2 calls)
  [test] anthropic/claude-haiku-4-5 × bndes ... ok (111

## Pipeline principal

Sobrescreve xlsx antigos. Erro numa combinação não para as outras. Sumário
no fim separa sucessos, parciais (estouro) e falhas.

In [ ]:
sucessos, parciais, falhas = [], [], []

for provider, modelo in MODELOS:
    for edital_nome, edital_id in EDITAIS:
        n = apagar_antigos(provider, modelo, edital_nome)
        if n:
            print(f"\n[del]  {n} xlsx antigo(s) de {provider}/{modelo} × {edital_nome}")
        print(f"\n[run]  {provider}/{modelo} × {edital_nome} (id={edital_id})")

        try:
            arq, erro_fatal = rodar_combo(provider, modelo, edital_nome, edital_id)
        except Exception as e:
            falhas.append((provider, modelo, edital_nome, str(e)))
            print(f"       ✗ FALHOU antes do xlsx: {e}")
            continue

        if erro_fatal:
            parciais.append((provider, modelo, edital_nome, arq, erro_fatal))
            print(f"       ⚠ PARCIAL — {erro_fatal}\n         salvo em {arq.name}")
        else:
            sucessos.append((provider, modelo, edital_nome, arq))
            print(f"       ✓ ok — salvo em {arq.name}")

print("\n" + "=" * 60)
print("=== SUMÁRIO ===")
print("=" * 60)
print(f"\n✓ Sucessos: {len(sucessos)}")
for p, m, e, a in sucessos:
    print(f"    {p}/{m} × {e}  →  {a.name}")
print(f"\n⚠ Parciais (estouro de orçamento): {len(parciais)}")
for p, m, e, a, err in parciais:
    print(f"    {p}/{m} × {e}  →  {a.name}\n        {err}")
print(f"\n✗ Falhas: {len(falhas)}")
for p, m, e, err in falhas:
    print(f"    {p}/{m} × {e}\n        {err}")


[run]  openai/gpt-4o-mini × bndes (id=1)
      [ 1/50]   ok   5.24s  calls=3  in= 8490 (cache   35%)  out= 135  acum=$0.0014
      [ 2/50]   ok   3.17s  calls=2  in= 6514 (cache   24%)  out=  67  acum=$0.0024
      [ 3/50]   ok   3.08s  calls=2  in= 7004 (cache   22%)  out=  94  acum=$0.0035
      [ 4/50]   ok    4.7s  calls=2  in= 6323 (cache    0%)  out= 189  acum=$0.0045
      [ 5/50]   ok    4.2s  calls=2  in= 7060 (cache    0%)  out= 179  acum=$0.0057
      [ 6/50]   ok   5.84s  calls=2  in= 7027 (cache    0%)  out= 239  acum=$0.0069
      [ 7/50]   ok   3.06s  calls=2  in= 7033 (cache    0%)  out=  96  acum=$0.0080
      [ 8/50]   ok   4.53s  calls=2  in= 6298 (cache    0%)  out= 104  acum=$0.0090
      [ 9/50]   ok   3.67s  calls=2  in= 6190 (cache    0%)  out= 135  acum=$0.0100
      [10/50]   ok   5.53s  calls=2  in= 6701 (cache   23%)  out= 210  acum=$0.0112
      [11/50]   ok   3.48s  calls=2  in= 6067 (cache   25%)  out= 132  acum=$0.0122
      [12/50]   ok   9.82s  calls=

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddc50-558e-79d3-b94b-405070846cec,id=019ddc50-558e-79d3-b94b-405070846cec; trace=019ddc50-5b20-7751-8895-63d83d16e0de,id=019ddc50-5b20-7751-8895-63d83d16e0de
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddc50-5b20-7751-8895-63d83d16e0de,id=019ddc50-5b20-7751-8895-63d83d16e0de; trace=019

      [42/50]   ok   5.12s  calls=4  in=23569 (cache   27%)  out= 273  acum=$0.0884


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddc50-6915-7591-812d-eb79997f9d4f,id=019ddc50-6915-7591-812d-eb79997f9d4f; trace=019ddc50-6f20-7e13-8965-9d0cf46cf92b,id=019ddc50-6f20-7e13-8965-9d0cf46cf92b
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddc50-6f20-7e13-8965-9d0cf46cf92b,id=019ddc50-6f20-7e13-8965-9d0cf46cf92b; trace=019

      [43/50]   ok   5.15s  calls=5  in=25573 (cache   62%)  out= 263  acum=$0.0953


...

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddca1-7304-7121-ad51-f13ccd5c2ebb,id=019ddca1-7304-7121-ad51-f13ccd5c2ebb; trace=019ddca1-7e7a-7c71-b25d-ee5f4b4e0fe4,id=019ddca1-7e7a-7c71-b25d-ee5f4b4e0fe4
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddca1-7e7a-7c71-b25d-ee5f4b4e0fe4,id=019ddca1-7e7a-7c71-b25d-ee5f4b4e0fe4; trace=019

      [49/50]   ok  18.09s  calls=2  in= 9944 (cache    0%)  out= 627  acum=$0.9896


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddca1-ab07-75f1-abfc-0a6b2f815c3d,id=019ddca1-ab07-75f1-abfc-0a6b2f815c3d; trace=019ddca1-c521-71e2-b4d3-2a99d4587839,id=019ddca1-c521-71e2-b4d3-2a99d4587839
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddca1-c521-71e2-b4d3-2a99d4587839,id=019ddca1-c521-71e2-b4d3-2a99d4587839; trace=019

      [50/50]   ok  11.87s  calls=2  in=10484 (cache    0%)  out= 641  acum=$1.0032
       ✓ ok — salvo em petrobras__anthropic__claude-haiku-4-5__20260430_012454.xlsx

=== SUMÁRIO ===

✓ Sucessos: 15
    openai/gpt-4o-mini × bndes  →  bndes__openai__gpt-4o-mini__20260429_231806.xlsx
    openai/gpt-4o-mini × cvm  →  cvm__openai__gpt-4o-mini__20260429_232228.xlsx
    openai/gpt-4o-mini × petrobras  →  petrobras__openai__gpt-4o-mini__20260429_232739.xlsx
    deepseek/deepseek-v4-flash × bndes  →  bndes__deepseek__deepseek-v4-flash__20260429_233504.xlsx
    deepseek/deepseek-v4-flash × cvm  →  cvm__deepseek__deepseek-v4-flash__20260429_234135.xlsx
    deepseek/deepseek-v4-flash × petrobras  →  petrobras__deepseek__deepseek-v4-flash__20260429_234903.xlsx
    openai/gpt-5.4-mini × bndes  →  bndes__openai__gpt-5.4-mini__20260429_235130.xlsx
    openai/gpt-5.4-mini × cvm  →  cvm__openai__gpt-5.4-mini__20260429_235347.xlsx
    openai/gpt-5.4-mini × petrobras  →  petrobras__openai__gpt-5.4-mini

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ddca1-da9f-7b32-8c69-8d1eaf6345c8,id=019ddca1-da9f-7b32-8c69-8d1eaf6345c8


## Sumário consolidado

In [19]:
import re

padrao_nome = re.compile(
    r"^(?P<edital>[^_]+)__(?P<provider>[^_]+)__(?P<modelo>.+?)__\d{8}_\d{6}(?:__PARCIAL)?\.xlsx$"
)

linhas = []
for arq in sorted(RESULTADOS_DIR.glob("*.xlsx")):
    m = padrao_nome.match(arq.name)
    if not m:
        continue
    df = pd.read_excel(arq)
    if "cache_read_tokens" not in df.columns:
        df["cache_read_tokens"] = 0
    if "custo_real_usd" not in df.columns:
        df["custo_real_usd"] = df["custo_usd"]

    linhas.append({
        "edital":           m["edital"],
        "provider":         m["provider"],
        "modelo":           m["modelo"],
        "n_perguntas":      len(df),
        "n_erros":          int(df["erro"].notna().sum()),
        "input_tokens":     int(df["input_tokens"].sum()),
        "output_tokens":    int(df["output_tokens"].sum()),
        "cache_read":       int(df["cache_read_tokens"].sum()),
        "cache_pct":        float(df["cache_read_tokens"].sum() / max(df["input_tokens"].sum(), 1)),
        "custo_estim_usd":  round(float(df["custo_usd"].sum()), 4),
        "custo_real_usd":   round(float(df["custo_real_usd"].sum()), 4),
        "latencia_med_s":   round(float(df["latencia_s"].mean()), 2),
        "latencia_p95_s":   round(float(df["latencia_s"].quantile(0.95)), 2),
        "arquivo":          arq.name,
    })

df_sumario = pd.DataFrame(linhas).sort_values(["edital", "custo_real_usd"]).reset_index(drop=True)
df_sumario["cache_pct"] = df_sumario["cache_pct"].map("{:.1%}".format)
df_sumario

,edital,provider,modelo,n_perguntas,n_erros,input_tokens,output_tokens,cache_read,cache_pct,custo_estim_usd,custo_real_usd,latencia_med_s,latencia_p95_s,arquivo
0,bndes,deepseek,deepseek-v4-flash,50,0,651794,21562,1223680,187.7%,0.0973,0.0405,8.89,17.08,bndes__deepseek__deepseek-v4-flash__20260429_2...
1,bndes,openai,gpt-4o-mini,50,0,423484,9378,60416,14.3%,0.0691,0.0646,5.45,10.38,bndes__openai__gpt-4o-mini__20260429_231806.xlsx
2,bndes,openai,gpt-5.4-mini,50,0,332401,8532,84864,25.5%,0.1002,0.0896,2.94,4.73,bndes__openai__gpt-5.4-mini__20260429_235130.xlsx
3,bndes,deepseek,deepseek-v4-pro,50,0,681518,23169,772096,113.3%,1.2665,0.2201,17.69,33.55,bndes__deepseek__deepseek-v4-pro__20260430_001...
4,bndes,anthropic,claude-haiku-4-5,50,0,700866,23247,0,0.0%,0.8171,0.8171,15.57,35.46,bndes__anthropic__claude-haiku-4-5__20260430_0...
5,cvm,deepseek,deepseek-v4-flash,50,0,558979,19510,715264,128.0%,0.0837,0.0262,7.82,13.25,cvm__deepseek__deepseek-v4-flash__20260429_234...
6,cvm,openai,gpt-4o-mini,50,0,433446,8765,145408,33.5%,0.0703,0.0594,5.23,8.58,cvm__openai__gpt-4o-mini__20260429_232228.xlsx
7,cvm,openai,gpt-5.4-mini,50,0,323919,8298,82816,25.6%,0.0976,0.0872,2.72,4.32,cvm__openai__gpt-5.4-mini__20260429_235347.xlsx
8,cvm,deepseek,deepseek-v4-pro,50,0,601746,22188,707584,117.6%,1.1242,0.1899,16.75,30.45,cvm__deepseek__deepseek-v4-pro__20260430_00250...
9,cvm,anthropic,claude-haiku-4-5,50,0,655949,21637,0,0.0%,0.7641,0.7641,15.80,36.02,cvm__anthropic__claude-haiku-4-5__20260430_010...


## Tabela cruzada — custo real por modelo × edital

In [20]:
df_pivot = (
    df_sumario.assign(modelo_full=lambda d: d["provider"] + "/" + d["modelo"])
              .pivot_table(index="modelo_full", columns="edital",
                          values="custo_real_usd", aggfunc="last")
)
df_pivot["total"] = df_pivot.sum(axis=1)
df_pivot.sort_values("total")

edital,bndes,cvm,petrobras,total
modelo_full,,,,
deepseek/deepseek-v4-flash,0.0405,0.0262,0.0359,0.1026
openai/gpt-4o-mini,0.0646,0.0594,0.0660,0.1900
openai/gpt-5.4-mini,0.0896,0.0872,0.1028,0.2796
deepseek/deepseek-v4-pro,0.2201,0.1899,0.2362,0.6462
anthropic/claude-haiku-4-5,0.8171,0.7641,1.0032,2.5844
